## Bài 1

In [3]:
from transformers import pipeline

def bai_1_fill_mask():
    print("--- BÀI 1: Masked Language Modeling ---")

    # Tải pipeline "fill-mask"
    mask_filler = pipeline("fill-mask", model="bert-base-uncased")

    # Câu đầu vào
    input_sentence = "Hanoi is the [MASK] of Vietnam."

    # Dự đoán
    predictions = mask_filler(input_sentence, top_k=5)

    print(f"Câu gốc: {input_sentence}")
    for i, pred in enumerate(predictions, 1):
        print(f"{i}. Từ dự đoán: '{pred['token_str']}' | Độ tin cậy: {pred['score']:.4f}")
        print(f"   -> Câu: {pred['sequence']}")

bai_1_fill_mask()

--- BÀI 1: Masked Language Modeling ---


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Câu gốc: Hanoi is the [MASK] of Vietnam.
1. Từ dự đoán: 'capital' | Độ tin cậy: 0.9991
   -> Câu: hanoi is the capital of vietnam.
2. Từ dự đoán: 'center' | Độ tin cậy: 0.0001
   -> Câu: hanoi is the center of vietnam.
3. Từ dự đoán: 'birthplace' | Độ tin cậy: 0.0001
   -> Câu: hanoi is the birthplace of vietnam.
4. Từ dự đoán: 'headquarters' | Độ tin cậy: 0.0001
   -> Câu: hanoi is the headquarters of vietnam.
5. Từ dự đoán: 'city' | Độ tin cậy: 0.0001
   -> Câu: hanoi is the city of vietnam.


## Bài 2

In [5]:
from transformers import pipeline, set_seed

def bai_2_text_generation():
    print("\n--- BÀI 2: Text Generation ---")

    # Tải pipeline "text-generation", sử dụng gpt2
    generator = pipeline("text-generation", model="gpt2")

    prompt = "The best thing about learning NLP is"
    set_seed(42)

    # Sinh văn bản
    output = generator(prompt, max_length=50, num_return_sequences=1, truncation=True)

    print(f"Prompt: '{prompt}'")
    print("-" * 30)
    print(output[0]['generated_text'])

bai_2_text_generation()


--- BÀI 2: Text Generation ---


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: 'The best thing about learning NLP is'
------------------------------
The best thing about learning NLP is it means that you can pick whatever you want—whether it's writing in-person or taking a video. Once I got to try NLP, I found a lot of interesting content I wanted to learn by


## Bài 3

In [8]:
import torch
from transformers import AutoTokenizer, AutoModel

def bai_3_sentence_embedding():
    print("\n--- BÀI 3: Sentence Representation (Mean Pooling) ---")

    # Chọn mô hình và tokenizer
    model_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    # Câu đầu vào
    sentences = ["This is a sample sentence"]
    # Tokenize
    inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
    # Shape sẽ là [1, 7] (1 câu, 7 tokens bao gồm [CLS] và [SEP])
    print("Input IDs shape:", inputs['input_ids'].shape)

    # Đưa qua mô hình (Forward pass)
    with torch.no_grad():
        outputs = model(**inputs)

    # Shape: (batch_size, sequence_length, hidden_size) -> (1, 7, 768)
    last_hidden_state = outputs.last_hidden_state

    # Thực hiện Mean Pooling
    # Tính trung bình vector của các token, nhưng bỏ qua token đệm (padding)
    attention_mask = inputs['attention_mask']

    # Mở rộng mask để khớp kích thước với hidden state
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()

    # Tính tổng các vector (chỉ tính những token có mask = 1)
    sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)

    # Tính tổng số lượng token thực (tránh chia cho 0 bằng cách dùng clamp)
    sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)

    # Phép chia để lấy trung bình
    sentence_embedding = sum_embeddings / sum_mask

    print("Kích thước vector biểu diễn câu:", sentence_embedding.shape)
    print("5 giá trị đầu tiên của vector:", sentence_embedding[0][:5])

bai_3_sentence_embedding()


--- BÀI 3: Sentence Representation (Mean Pooling) ---
Input IDs shape: torch.Size([1, 7])
Kích thước vector biểu diễn câu: torch.Size([1, 768])
5 giá trị đầu tiên của vector: tensor([-0.2424, -0.3832, -0.0138, -0.2991, -0.2145])
